# The Crucible — Colab T4 Test Harness

Runs the **real `crucible` package** end-to-end on a free Colab **T4 GPU** (the vision models need CUDA; local CPU is slow). This notebook does *not* duplicate the pipeline — it clones the repo and imports it, so it always tests the actual code.

**Before running:**
1. `Runtime → Change runtime type → Hardware accelerator: T4 GPU`.
2. Add these Colab **Secrets** (🔑 in the left sidebar, with notebook access enabled):
   - `GEMINI_API_KEY` — from https://aistudio.google.com/app/apikey
   - `KAGGLE_USERNAME` and `KAGGLE_KEY` — from https://www.kaggle.com/settings → API
3. The working branch must be pushed to GitHub so Colab can clone it.

Then `Runtime → Run all`.

In [ ]:
# 1. Clone the repo + install deps.
#    NOTE: we install the explicit list (NOT `-r requirements.txt`) so Colab's
#    preinstalled CUDA build of torch is kept rather than reinstalled.
!git clone --branch refactor/moondream https://github.com/SuperaNova/Crucible.git
%cd Crucible
!pip install -q transformers==4.51.3 google-genai google-adk gradio pillow numpy requests python-dotenv kaggle nest_asyncio sentencepiece protobuf
print('\nInstall complete.')

In [ ]:
# 2. GPU check — confirm the T4 is attached.
!nvidia-smi -L
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device       :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (vision models will be slow — switch to a T4)')

In [ ]:
# 3. Secrets — Kaggle creds (for the dataset) + Gemini key (for the Master Smith).
import os, json
from pathlib import Path
from google.colab import userdata

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / 'kaggle.json').write_text(json.dumps({
    'username': userdata.get('KAGGLE_USERNAME'),
    'key': userdata.get('KAGGLE_KEY'),
}))
(kaggle_dir / 'kaggle.json').chmod(0o600)

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['GOOGLE_API_KEY'] = os.environ['GEMINI_API_KEY']
print('Secrets configured.')

In [ ]:
# 3b. (optional) Sanity-check the Gemini key BEFORE running the pipeline.
#     Never prints the key. A 400 here = bad key (regenerate at AI Studio).
import os, requests
k = os.environ['GOOGLE_API_KEY']
print('prefix:', repr(k[:6]), '| len:', len(k), '| has whitespace:', k != k.strip())
r = requests.get('https://generativelanguage.googleapis.com/v1beta/models', params={'key': k.strip()})
print('HTTP', r.status_code, '\u2014', 'key OK' if r.status_code == 200 else r.text[:200])

In [ ]:
# 4. Download + Smelt the dataset (-> data/sprites_clean.npy). Same path as local setup.
!python setup.py

In [ ]:
# 5. Load the clean sprites and preview a random grid — pick two indices to fuse.
import numpy as np, random
import matplotlib.pyplot as plt

sprites = np.load('data/sprites_clean.npy')
print('Clean sprites:', sprites.shape)

sample = random.sample(range(len(sprites)), 32)
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for ax, i in zip(axes.flat, sample):
    ax.imshow(sprites[i][..., :3], interpolation='nearest')
    ax.set_title(f'#{i}', fontsize=7); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 6. Forge two items. This runs the REAL pipeline:
#    Stage 1 SigLIP identity + Moondream appearance -> Stage 2 part-level Master
#    Smith (ADK) -> Stage 3 deterministic prompt assembly + Flux + pixel post-proc.
import nest_asyncio; nest_asyncio.apply()
import pandas as pd
from IPython.display import display
from crucible import Forge

IDX_A = 0    # <-- choose from the grid above
IDX_B = 10   # <--

forge = Forge()
preview_a, preview_b, forged, meta = forge.quench(sprites[IDX_A], sprites[IDX_B])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [preview_a, forged, preview_b],
    [f"A: {meta['item_a']}", meta['fused_name'], f"B: {meta['item_b']}"],
):
    ax.imshow(img); ax.set_title(title, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

print('Archetype       :', meta['archetype'])
print('Structure anchor: Item', meta['structure_source'])
print('Lore            :', meta['reasoning'])
print('\nPart-level material blueprint:')
display(pd.DataFrame(meta['parts'], columns=['part', 'material', 'color', 'source', 'detail']))
print('\nAssembled Flux prompt:\n', meta['image_prompt'])

### Iterating

- Change `IDX_A` / `IDX_B` above and re-run cell 6 (each model reloads per call, then unloads to stay within VRAM).
- After pushing new commits to the branch, pull them without re-cloning:
  ```python
  !git pull
  ```
  then restart the runtime (`Runtime → Restart`) and re-run from cell 2 so the updated package is re-imported.
- Item identity comes from SigLIP zero-shot over `ITEM_VOCAB` in `crucible/autoencoder_appraiser.py` — edit that list if your sprites need different categories.